In [3]:
%pip install xgboost lightgbm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, roc_auc_score,
                              RocCurveDisplay, average_precision_score,
                              confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# Load processed splits
train = pd.read_csv('../data/processed/train.csv')
val   = pd.read_csv('../data/processed/val.csv')

FEATURES = [
    'GridPosition', 'QualiGapToPole_pct', 'IsWet',
    'SafetyCarDeployed', 'AvgFinish_Last3', 'DNF_Rate_Last5',
    'PodiumRate_Last5', 'Constructor_AvgPts_Last3', 'SeasonProgress'
]
TARGET = 'Podium'

X_train = train[FEATURES]
y_train = train[TARGET]
X_val   = val[FEATURES]
y_val   = val[TARGET]

# Class imbalance ratio — needed for XGBoost
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

print(f"Train: {len(X_train)} rows | Podium rate: {y_train.mean():.2%}")
print(f"Val:   {len(X_val)} rows  | Podium rate: {y_val.mean():.2%}")
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

Note: you may need to restart the kernel to use updated packages.
Train: 1120 rows | Podium rate: 15.00%
Val:   360 rows  | Podium rate: 15.00%
scale_pos_weight: 5.67


In [4]:
# Same preprocessor used inside all three model pipelines
# Ensures identical treatment of features across all models
preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

print("Preprocessor pipeline defined")
print("Steps: median imputation → standard scaling")

Preprocessor pipeline defined
Steps: median imputation → standard scaling


In [5]:
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ))
])

lr_pipeline.fit(X_train, y_train)
print("Logistic Regression trained")

Logistic Regression trained
